# Gold — Chuẩn hóa & Tiền xử lý văn bản tiếng Việt (NLP Preprocessing)

Theo phương pháp luận (Step 3 của bài báo gốc):
- **Lowercasing**: Viết thường khối văn bản.
- **Punctuation normalization**: Chuẩn hóa dấu câu (Loại bỏ các ký hiệu đặc biệt, chỉ giữ lại chữ tiếng Việt).
- **Tokenization & Lemmatization**: Trong NLP tiếng Việt, khái niệm tương đương của Lemmatization (quy về từ gốc) chính là **Word Segmentation (Tách từ)**. Ví dụ: biến `ngân hàng` thành cụm có nghĩa `ngân_hàng` thay vì 2 ký tự rời rạc vô nghĩa. Ta dùng thư viện `pyvi` cho chuẩn hóa tiếng Việt.
- **Stopwords removal**: Loại bỏ các từ vô nghĩa, từ liên kết (như "và", "là", "thì", "mà",...) dựa trên bộ từ điển tiếng Việt chuẩn mã nguồn mở.

**Input**: `data/<YEAR>/<BANK>/silver/3/*.json`
**Output**: `data/<YEAR>/<BANK>/gold/preprocessed_*.json`

In [1]:
# Cài đặt thư viện tách từ pyvi (Python Vietnamese) và requests
%pip -q install pyvi requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
# --- CẤU HÌNH NĂM VÀ NGÂN HÀNG CẦN XỬ LÝ ---
TARGET_YEAR = "2024"
TARGET_BANK = "PVcomBank"  # Đổi thành "vietcombank", "bidv", v.v. hoặc "*" nếu muốn chạy tất cả ngân hàng trong năm


In [3]:
import json
import re
from pathlib import Path

import requests
from pyvi import ViTokenizer

In [4]:
# 1. Cấu hình thư mục

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() in {"thread_1", "thread_2"}:
    pass

DATA_ROOT = PROJECT_ROOT / "data" / str(TARGET_YEAR)
STAGE_IN = "silver/3"
STAGE_OUT = "gold"

In [5]:
# 2. Tải danh sách Stopwords Tiếng Việt từ Github
def load_vietnamese_stopwords() -> set:
    url = "https://raw.githubusercontent.com/stopwords/vietnamese-stopwords/master/vietnamese-stopwords.txt"
    try:
        resp = requests.get(url)
        resp.raise_for_status()
        
        # Đặc biệt: ViTokenizer của pyvi nối từ ghép bằng dấu gạch dưới (VD: "ngân_hàng")
        # Ta cần xử lý danh sách Stopwords cũng nối bằng '_' để khớp với hàm tách từ.
        words = [w.strip().replace(" ", "_") for w in resp.text.splitlines() if w.strip()]
        return set(words)
    except Exception as e:
        print("Lỗi khi tải stopwords:", e)
        return set()
        
VI_STOPWORDS = load_vietnamese_stopwords()
print(f"Đã tải {len(VI_STOPWORDS)} stopwords tiếng Việt.")
# Lấy thử 10 stopwords làm mẫu
print("Sample:", list(VI_STOPWORDS)[:10])

Đã tải 1942 stopwords tiếng Việt.
Sample: ['tại_sao', 'ra_sao', 'hơn_trước', 'mỗi_một', 'lời', 'quay_bước', 'về_phần', 'thì_phải', 'ắt_là', 'em']


In [6]:
# 3. Hàm Tiền xử lý (Preprocessing) phù hợp với tiếng Việt
def preprocess_vietnamese_text(text: str) -> str:
    if not text or not isinstance(text, str):
        return ""
    
    # 1. Lowercasing: Viết thường
    text = text.lower()
    
    # 2. Punctuation normalization: Xóa dấu câu (giữ lại các ký tự alpha-numeric và khoảng trắng)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 3 & 4. Tokenization (Tách từ ghép)
    # Hàm ViTokenizer.tokenize("ngân hàng ABC") -> "ngân_hàng ABC"
    tokenized_text = ViTokenizer.tokenize(text)
    
    # 5. Stopwords removal: Loại bỏ các từ bị thừa
    tokens = tokenized_text.split()
    filtered_tokens = [t for t in tokens if t not in VI_STOPWORDS]
    
    return " ".join(filtered_tokens)

# Test thử một câu
sample_text = "Ngân hàng thương mại Vietcombank công bố báo cáo phát triển bền vững ESG rất tuyệt vời, nhưng cũng đầy chông gai!"
print("Gốc:", sample_text)
print("Sau xử lý (Tiếng Việt phân mảnh + Stopwords Removed):")
print(preprocess_vietnamese_text(sample_text))

Gốc: Ngân hàng thương mại Vietcombank công bố báo cáo phát triển bền vững ESG rất tuyệt vời, nhưng cũng đầy chông gai!
Sau xử lý (Tiếng Việt phân mảnh + Stopwords Removed):
ngân_hàng thương_mại vietcombank công_bố báo_cáo phát_triển bền_vững esg tuyệt_vời chông_gai


In [7]:
# 4. Đọc file Json từ Silver 3, tiền xử lý content và lưu vào Gold
def run_preprocessing_pipeline():
    bank_path = "**" if TARGET_BANK == "*" else TARGET_BANK
    input_files = list(DATA_ROOT.glob(f"{bank_path}/{STAGE_IN}/*.json"))
    print(f"Tìm thấy {len(input_files)} file Json trong {STAGE_IN} của ngân hàng {TARGET_BANK}\n")
    
    for file_path in input_files:
        # Đường dẫn gốc: data/<YEAR>/<BANK>/silver/3/<tên file>
        # => Lùi về 2 cấp thư mục cha (parents[2]) sẽ là folder <BANK>
        bank_dir = file_path.parents[2] 
        
        try:
            data = json.loads(file_path.read_text(encoding="utf-8"))
        except Exception as e:
            print(f"Lỗi json tại {file_path}: {e}")
            continue
            
        out_data = {}
        for bank_name, items in data.items():
            if not isinstance(items, list):
                continue
            
            processed_items = []
            for item in items:
                # Copy toàn bộ thông tin (title, link, original_content, quotes...) từ silver/3 sang
                new_item = item.copy()
                
                # Tiền xử lý văn bản từ trường "content" bằng bộ NLP tiếng Việt
                content = new_item.get("content", "")
                
                # Lưu dưới 1 trường mới để đối chiếu nếu cần, không đè vỡ content cũ
                # Thêm try/except để tránh lỗi với nội dung null
                try:
                    new_item["preprocessed_content"] = preprocess_vietnamese_text(content)
                except Exception as e:
                    print(f"Lỗi khi xử lý text: {e}")
                    new_item["preprocessed_content"] = ""
                
                processed_items.append(new_item)
                
            out_data[bank_name] = processed_items
        
        # Khởi tạo thư mục Gold
        out_dir = bank_dir / STAGE_OUT
        out_dir.mkdir(parents=True, exist_ok=True)
        
        # Đặt tên file bằng cách thêm chữ preprocessed_ ở đầu
        out_path = out_dir / f"preprocessed_{file_path.name}"
        
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(out_data, f, ensure_ascii=False, indent=2)
            
        print(f"Đã xử lý xong: {file_path.name}")
        print(f" -> Lưu tại: {out_path}\n")

run_preprocessing_pipeline()
print("Hoàn tất quy trình Preprocessing!")

Tìm thấy 1 file Json trong silver/3 của ngân hàng PVcomBank

Đã xử lý xong: esg_check_llms_20260405_085400.json
 -> Lưu tại: d:\NCKH\Thread_1\data\2024\PVcomBank\gold\preprocessed_esg_check_llms_20260405_085400.json

Hoàn tất quy trình Preprocessing!
